# Trajectory profiling reports

Analyze existing Scalene profiles and library traces for Spaceship Titanic, House Price, NYC Housing, and NYC Taxi. This notebook does **not** execute or train pipelines.

All category calculations come from `scripts/report_categories.py`; hotspot calculations and labels come from `scripts/report_hotspot.py`. Run all cells to regenerate CSVs and plots. Each trajectory gets its own figure so labels remain readable.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "scripts/report_hotspot.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import report_categories, report_hotspot

TRAJECTORIES = [
    "spaceship_titanic_students",
    "house_price_students",
    "nyc_housing_students",
    "nyc_taxi_students",
]
TOP_N = 10


## Input coverage

Require one profile and one library trace for every pipeline in each manifest. Interrupted-run backups ending in `.json.bak` are excluded. File coverage alone cannot establish that a run finished; these datasets were checked against completion logs before analysis.

In [ ]:
coverage = []
for name in TRAJECTORIES:
    manifest = json.loads((ROOT / "trajectories" / name / "trajectory.json").read_text())
    expected = {Path(p["file"]).stem for step in manifest["steps"] for p in step["pipelines"]}
    counts = {}
    for kind in ("profiles", "libraries"):
        found = {p.stem for p in (ROOT / "profiling" / kind / name).glob("pipeline_*.json")}
        if found != expected:
            raise ValueError(f"{name}/{kind}: missing={sorted(expected - found)}, extra={sorted(found - expected)}")
        counts[kind] = len(found)
    coverage.append({"trajectory": name, "expected_pipelines": len(expected), **counts})
display(pd.DataFrame(coverage).set_index("trajectory"))


## Category reports

Each pipeline is split into **setup**, **data handling**, **models**, and **other**, using the existing module-prefix rules. Unmatched non-import lines fall back to data handling; this is a heuristic, not a semantic classification of every library. Code outside the pipeline file is assigned to other.

`total_s` includes Python, native, and system estimates. CSV `share_of_runtime` uses elapsed profiled runtime as its denominator; the plot's percentage panel normalizes to the sum of attributed category time. Peaks are not additive and cannot directly determine concurrency limits. Allocation and copy quantities are volumes, not durations.

Outputs: `profiling/results/<trajectory>/categories.csv` and `categories.png`.

In [ ]:
categories_by_trajectory = {}
for name in TRAJECTORIES:
    totals = report_categories.analyze_trajectory(name)
    categories_by_trajectory[name] = totals
    output_dir = ROOT / "profiling/results" / name
    output_dir.mkdir(parents=True, exist_ok=True)
    totals.to_csv(output_dir / "categories.csv", index=False)
    display(Markdown(f"### {name}"))
    display(totals.pivot(index="pipeline", columns="category", values="total_s")
            [report_categories.CATEGORIES].round(3))
    fig = report_categories.plot(totals, output_dir / "categories.png")
    display(fig)
    plt.close(fig)


## Top 10 hotspots across each trajectory

Statements are grouped by their full source text after removing indentation and decoding Scalene Unicode escapes. Different parameters or variable names remain separate. Short plot labels are for display only; they do not change grouping. Identical statements can represent different models or inputs and do not establish cacheability.

Only lines in each pipeline file are ranked. Function timings are not added to line timings. Python, native, and system estimates are retained; other files and unattributed residual time are reported separately. Percentages use summed profiled runtime, not unprofiled benchmark time.

Outputs: `results/<trajectory>/hotspots.csv` (full ranking), `hotspots_details.csv` (all locations behind Top N), and `hotspots.png`.

In [ ]:
analyses = report_hotspot.analyze_all(TRAJECTORIES, top_n=TOP_N)
top10_details_by_trajectory = {}
for name, analysis in analyses.items():
    details = report_hotspot.top10_details(analysis)
    top10_details_by_trajectory[name] = details
    output_dir = ROOT / "results" / name
    output_dir.mkdir(parents=True, exist_ok=True)
    analysis.operation_summary.to_csv(output_dir / "hotspots.csv", index=False)
    details.to_csv(output_dir / "hotspots_details.csv", index=False)
    display(Markdown(f"### {name}"))
    report_hotspot.report_trajectory(analysis)
    display(analysis.top10.round(3))
    fig = report_hotspot.plot_hotspots({name: analysis}, output_dir / "hotspots.png")
    display(fig)
    plt.close(fig)

# Preserve the original Spaceship Titanic aliases for interactive exploration.
spaceship = analyses["spaceship_titanic_students"]
house_price = analyses["house_price_students"]
nyc_housing = analyses["nyc_housing_students"]
nyc_taxi = analyses["nyc_taxi_students"]
line_details, runs = spaceship.line_details, spaceship.runs
operation_summary, top10 = spaceship.operation_summary, spaceship.top10
trajectory_profile_s = spaceship.profile_s
top10_details = top10_details_by_trajectory["spaceship_titanic_students"]


## Report overview

Shares below use each trajectory's summed profiled elapsed time. Category shares can differ from 100% because sampling attribution may leave a residual. The trajectories have different data, models, and workload sizes; their total durations are not a controlled performance comparison.

In [ ]:
overview = []
for name, analysis in analyses.items():
    category_seconds = categories_by_trajectory[name].groupby("category")["total_s"].sum()
    overview.append({
        "trajectory": name,
        "pipelines": len(analysis.runs),
        "profile_s": analysis.profile_s,
        "top_10_share_pct": analysis.top10.share_of_runs_pct.sum(),
        **{f"{category}_pct": category_seconds[category] / analysis.profile_s * 100
           for category in report_categories.CATEGORIES},
    })
report_overview = pd.DataFrame(overview).set_index("trajectory")
report_overview.to_csv(ROOT / "results/report_overview.csv")
display(report_overview.round(3))


## Traceable hotspot details

Every source location behind Top N is retained, including locations with zero sampled time. `n_pipelines` counts source presence, not just timed execution. Select any trajectory below; filter the full table by rank or pipeline to inspect a candidate.

In [ ]:
DETAIL_TRAJECTORY = "nyc_taxi_students"
details = top10_details_by_trajectory[DETAIL_TRAJECTORY]
display(details.drop(columns=["source_file", "source"]).head(30).round(4))
# Examples:
# details.query("rank == 1")
# top10_details_by_trajectory["nyc_housing_students"].query("pipeline == 'pipeline_1_1'")


## Interpretation and next experiments

- **Spaceship Titanic:** training and first-time library imports dominate the leading statements. Check model and fold identity before proposing reuse.
- **House Price:** model fitting and CSV loading lead the ranking. Evaluate data-loading reuse and training improvements separately.
- **NYC Housing:** the leading source statements include parquet loading and sampling. Examine repeated table reads and transformations; a sampled row subset does not eliminate the cost of reading the original tables.
- **NYC Taxi:** `model.fit(X_train, y_train)` accounts for approximately 98.3% of summed profiled runtime in this dataset. It combines different estimators, feature branches, and folds; it is not one reusable computation.

These are priorities for investigation, not measured speedups. Validate any changes with repeated unprofiled runs and predictive-quality checks. System time is not a direct disk-I/O measurement, and Python/native shares alone do not prove that parallelism or vectorization will help.
